**Установка зависимостей**



In [1]:
!pip install ultralytics opencv-python numpy lap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.8 MB/s eta 0:00:00


**Загрузка видео**

In [2]:
from google.colab import files
import os

uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"Загружен файл: {video_filename}")

Saving 12345.mp4 to 12345.mp4
Загружен файл: 12345.mp4


**Настройка параметров**

In [3]:
direction = 'top_to_bottom'  # @param ["left_to_right", "top_to_bottom"]
line_coords = "0,800,0,800"  # @param {type:"string"}   для горизонтальной: 0,Y,0,Y; для вертикальной: X,0,X,0


target_fps = 10
target_width = 1920
target_height = 1080
conf = 0.5
iou = 0.5

**Запуск обработки**

In [4]:
# Запуск подсчёта
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import clear_output

# Функция проверки пересечения отрезков
def line_intersection(p1, p2, p3, p4):
    def ccw(a, b, c):
        return (c[1]-a[1])*(b[0]-a[0]) > (b[1]-a[1])*(c[0]-a[0])
    return ccw(p1,p3,p4) != ccw(p2,p3,p4) and ccw(p1,p2,p3) != ccw(p1,p2,p4)

# Парсим линию
x1,y1,x2,y2 = map(int, line_coords.split(','))
pt1_in, pt2_in = (x1,y1), (x2,y2)

# Загружаем модель
model = YOLO('yolov8n.pt')

# Открываем видео
cap = cv2.VideoCapture(video_filename)
orig_fps = cap.get(cv2.CAP_PROP_FPS)
skip_frames = max(1, int(orig_fps / target_fps))

w_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Исходное видео: {w_orig}x{h_orig}, {orig_fps} FPS")

# Подготовка выходного видео
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_path = 'output_video.mp4'
out = cv2.VideoWriter(output_path, fourcc, target_fps, (target_width, target_height))

# Инициализация линии
if direction == 'left_to_right':
    line_x = pt1_in[0]
    line_pt1, line_pt2 = (line_x, 0), (line_x, target_height)
elif direction == 'top_to_bottom':
    line_y = pt1_in[1]
    line_pt1, line_pt2 = (0, line_y), (target_width, line_y)
else:
    raise ValueError("direction must be 'left_to_right' or 'top_to_bottom'")

counted_ids = set()
total_crossings = 0
prev_centers = {}

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    if frame_idx % skip_frames != 0:
        frame_idx += 1
        continue

    # Изменяем разрешение
    frame = cv2.resize(frame, (target_width, target_height))

    # Трекинг
    results = model.track(frame, persist=True, conf=conf, iou=iou, verbose=False)
    boxes = results[0].boxes

    annotated = frame.copy()

    if boxes is not None and boxes.id is not None:

        vehicle_classes = [2, 3, 5, 7]
        mask = np.isin(boxes.cls.cpu().numpy(), vehicle_classes)

        vehicle_boxes = boxes.xyxy[mask].cpu().numpy()
        vehicle_ids = boxes.id[mask].cpu().numpy().astype(int)
        vehicle_cls = boxes.cls[mask].cpu().numpy().astype(int)

        for box, obj_id, cls_id in zip(vehicle_boxes, vehicle_ids, vehicle_cls):
            x1b, y1b, x2b, y2b = map(int, box)
            center = ((x1b + x2b)//2, (y1b + y2b)//2)

            color = (255, 0, 0) if obj_id in counted_ids else (0, 255, 0)
            cv2.rectangle(annotated, (x1b, y1b), (x2b, y2b), color, 2)
            label = f'{model.names[cls_id]} ID:{obj_id}'
            cv2.putText(annotated, label, (x1b, y1b-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            cv2.circle(annotated, center, 4, (0, 255, 255), -1)

            # Проверка пересечения
            if obj_id in prev_centers:
                prev = prev_centers[obj_id]
                if line_intersection(prev, center, line_pt1, line_pt2):
                    cross_ok = False
                    if direction == 'left_to_right' and prev[0] < line_x <= center[0]:
                        cross_ok = True
                    elif direction == 'top_to_bottom' and prev[1] < line_y <= center[1]:
                        cross_ok = True
                    if cross_ok and obj_id not in counted_ids:
                        total_crossings += 1
                        counted_ids.add(obj_id)
            prev_centers[obj_id] = center

    # Рисуем линию
    cv2.line(annotated, line_pt1, line_pt2, (0, 0, 255), 3)
    cv2.putText(annotated, f'Crossings: {total_crossings}', (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    out.write(annotated)
    frame_idx += 1

    # Прогресс каждые 100 кадров
    if frame_idx % 100 == 0:
        clear_output(wait=True)
        print(f"Обработано кадров: {frame_idx}, пересечений: {total_crossings}")

cap.release()
out.release()
cv2.destroyAllWindows()
clear_output(wait=True)
print(f"✅ Готово! Всего пересечений: {total_crossings}")

✅ Готово! Всего пересечений: 10


**Просмотр результата**

In [6]:
from IPython.display import Video
Video(output_path, width=960)

**Скачать видео**

In [7]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>